In [1]:
from joblib import load
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [2]:
# Step 1: Load data
aggregated_metrics = load("/home/hanwu/AnalogDesignAuto/AnalogDesignAuto_MultiAgent/custom_env/run_test/sum_specs_3.joblib")  # Adjust the path as necessary

# Step 2: Data cleaning
df = pd.DataFrame(aggregated_metrics)
print(f"Original data groups: {df.shape[0]}")

# New logic for dynamic data cleaning
for column in df.columns:
    original_count = df.shape[0]
    if column == 'pwr':
        # Delete rows where 'pwr' is 1
        df = df[df[column] != 1]
    else:
        # Delete rows where other columns are 0
        df = df[df[column] != 0]
    cleaned_count = df.shape[0]
    print(f"Deleted {original_count - cleaned_count} rows from '{column}' due to cleaning criteria.")

# Print the original and cleaned data counts for comparison
print(f"Cleaned data groups: {df.shape[0]}")

Original data groups: 513391
Deleted 7480 rows from 'phaseMargin' due to cleaning criteria.
Deleted 0 rows from 'gainBandWidth' due to cleaning criteria.
Deleted 412 rows from 'pwr' due to cleaning criteria.
Cleaned data groups: 505499


In [3]:
# Step 3: Display data distribution
fig, axes = plt.subplots(1, len(df.columns), figsize=(5 * len(df.columns), 5))
for i, column in enumerate(df.columns):
    axes[i].hist(df[column], bins=10, color='skyblue', edgecolor='black')
    axes[i].set_title(f'{column} Distribution')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

# Step 4: Interactive analysis
def update_charts(**ranges):
    conditions = [(df[column] >= ranges[f"{column}_range"][0]) & (df[column] <= ranges[f"{column}_range"][1]) for column in df.columns]
    filtered_df = df[np.all(conditions, axis=0)]
    
    if not filtered_df.empty:
        fig, axes = plt.subplots(1, len(df.columns), figsize=(5 * len(df.columns), 5))
        for i, column in enumerate(df.columns):
            axes[i].hist(filtered_df[column], bins=10, color='skyblue', edgecolor='black')
            axes[i].set_title(f'Filtered {column} Distribution')
            axes[i].set_xlabel('Value')
            axes[i].set_ylabel('Frequency')
        plt.tight_layout()
        plt.show()
    else:
        print("No data available within the specified range. Please adjust the filters.")

# Generate sliders for each column dynamically
sliders = {}
for column in df.columns:
    if df[column].dtype == float:
        slider = widgets.FloatRangeSlider(
            value=[df[column].min(), df[column].max()],
            min=df[column].min(), max=df[column].max(),
            step=0.1, description=f'{column}:'
        )
    else:  # Assuming integer values for simplicity
        slider = widgets.IntRangeSlider(
            value=[df[column].min(), df[column].max()],
            min=df[column].min(), max=df[column].max(),
            step=1, description=f'{column}:'
        )
    sliders[f"{column}_range"] = slider

# Display the widgets and set up the interaction
for slider in sliders.values():
    display(slider)
widgets.interactive(update_charts, **sliders)

TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]

interactive(children=(FloatRangeSlider(value=(3.50619, 179.7261), description='phaseMargin:', max=179.7261, mi…